In [ ]:
## Plot 1

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import html as html_lib

# ==========================================
# 1. Preprocess data
# ==========================================
df = pd.read_csv('Micro_With_VDem.csv', parse_dates=['event_date'])
target_countries = ['Germany', 'France', 'United Kingdom', 'Czech Republic', 'Slovakia']
df = df[df['country'].isin(target_countries)].copy()

df['Year'] = df['event_date'].dt.year
df['Quarter'] = df['event_date'].dt.quarter
df['Time_Quarter'] = df['Year'].astype(str) + '-Q' + df['Quarter'].astype(str)

df_agg = df.groupby(['country', 'location', 'latitude', 'longitude', 'Time_Quarter']).agg(
    total_protests=('event_id_cnty', 'count'),
    repressed_count=('sub_event_type', lambda x: x.isin([
        'Protest with intervention',
        'Excessive force against protesters'
    ]).sum()),
    vdem_libdem=('v2x_libdem', 'mean'),
    vdem_rule=('v2x_rule', 'mean')
).reset_index()

# ==========================================
# 2. Cleaning and classification
# ==========================================
df_map_filtered = df_agg[df_agg['total_protests'] >= 3].copy()

def classify_intervention(count):
    if count == 0:
        return '0: No Intervention'
    elif count <= 3:
        return '1-3: Low Intervention'
    else:
        return '4+: High Intervention'

df_map_filtered['Intervention_Level'] = df_map_filtered['repressed_count'].apply(classify_intervention)

color_map = {
    "0: No Intervention": "#AED6F1",
    "1-3: Low Intervention": "#F5B041",
    "4+: High Intervention": "#E74C3C"
}

quarters = sorted(df_map_filtered['Time_Quarter'].unique())
q0 = quarters[0]

# ==========================================
# 3. Build the figure
# ==========================================
fig = go.Figure()
df_q0_all = df_map_filtered[df_map_filtered['Time_Quarter'] == q0]

# Trace 0: All countries view
fig.add_trace(go.Scattermapbox(
    lat=df_q0_all['latitude'],
    lon=df_q0_all['longitude'],
    mode='markers',
    name='All Countries',
    showlegend=False,
    marker=dict(
        size=df_q0_all['total_protests'] * 1.2,
        sizemin=4,  # Prevent scatter points be ignored with excessively small values 
        color=df_q0_all['Intervention_Level'].map(color_map),
        opacity=0.85
    ),
    customdata=df_q0_all[['location', 'country', 'total_protests', 'repressed_count', 'vdem_libdem', 'vdem_rule']],
    hovertemplate=(
        "<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
        "Total Protests: %{customdata[2]}<br>"
        "Interventions: %{customdata[3]}<br>"
        "<b>LibDem:</b> %{customdata[4]:.3f} | <b>Rule of Law:</b> %{customdata[5]:.3f}"
        "<extra></extra>"
    ),
    visible=True
))

# Traces 1-5: Single-country views
for i, c in enumerate(target_countries):
    df_q0_c = df_q0_all[df_q0_all['country'] == c]
    fig.add_trace(go.Scattermapbox(
        lat=df_q0_c['latitude'],
        lon=df_q0_c['longitude'],
        mode='markers',
        name=c,
        showlegend=False,
        marker=dict(
            size=df_q0_c['total_protests'] * 1.2,
            sizemin=4, 
            color=df_q0_c['Intervention_Level'].map(color_map),
            opacity=0.85
        ),
        customdata=df_q0_c[['location', 'country', 'total_protests', 'repressed_count', 'vdem_libdem', 'vdem_rule']],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Total Protests: %{customdata[2]}<br>"
            "Interventions: %{customdata[3]}<br>"
            "<b>LibDem:</b> %{customdata[4]:.3f}"
            "<extra></extra>"
        ),
        visible=False
    ))

# ==========================================
# 4. Precompute animation frames
# ==========================================
all_frames = []

for q in quarters:
    df_q_all = df_map_filtered[df_map_filtered['Time_Quarter'] == q]
    frame_data = []

    frame_data.append(go.Scattermapbox(
        lat=df_q_all['latitude'],
        lon=df_q_all['longitude'],
        marker=dict(
            size=df_q_all['total_protests'] * 1.2,
            sizemin=4, 
            color=df_q_all['Intervention_Level'].map(color_map)
        ),
        customdata=df_q_all[['location', 'country', 'total_protests', 'repressed_count', 'vdem_libdem', 'vdem_rule']]
    ))

    for c in target_countries:
        df_q_c = df_q_all[df_q_all['country'] == c]
        frame_data.append(go.Scattermapbox(
            lat=df_q_c['latitude'],
            lon=df_q_c['longitude'],
            marker=dict(
                size=df_q_c['total_protests'] * 1.2,
                sizemin=4, 
                color=df_q_c['Intervention_Level'].map(color_map)
            ),
            customdata=df_q_c[['location', 'country', 'total_protests', 'repressed_count', 'vdem_libdem', 'vdem_rule']]
        ))

    all_frames.append(go.Frame(data=frame_data, name=q))

fig.frames = all_frames

# ==========================================
# 5. Controls and layout
# ==========================================
country_buttons = [
    dict(label="All Countries", method="restyle",
         args=[{"visible": [True, False, False, False, False, False]}]
    )
]

for i in range(len(target_countries)):
    vis = [False] * 6
    vis[i + 1] = True
    country_buttons.append(
        dict(
            label=target_countries[i],
            method="restyle",
            args=[{"visible": vis}]
        )
    )

custom_legend_title = (
    "<span style='font-size:18px;'><b>Spatial Distribution of Protests and State Repressive Intervention Hotspots in Five European Countries</b><br>"
        
    "<span style='display:inline-block;width:12px;height:12px;background-color:#666;border-radius:50%;margin-right:4px;'></span>"
    "<span style='color:#AED6F1; font-size:14px;'>No Intervention</span> &nbsp;&nbsp;&nbsp;&nbsp;"
    
    "<span style='display:inline-block;width:12px;height:12px;background-color:#666;border-radius:50%;margin-right:4px;'></span>"
    "<span style='color:#F5B041; font-size:14px;'>Low Intervention (1-3)</span> &nbsp;&nbsp;&nbsp;&nbsp;"
    
    "<span style='display:inline-block;width:12px;height:12px;background-color:#666;border-radius:50%;margin-right:4px;'></span>"
    "<span style='color:#E74C3C; font-size:14px;'>High Intervention (4+)</span> &nbsp;&nbsp;&nbsp;&nbsp;"
    
    "<span style='font-size:14px; color:#666;'>| Marker size indicates total protests; hover for V-Dem scores</span>"
    "</sup>"
)

fig.update_layout(
    uirevision='locked',
    updatemenus=[
        dict(buttons=country_buttons, direction="down", x=0.0, y=1,
             xanchor="left", yanchor="top", showactive=True, bgcolor="white",
             bordercolor="#ccc"
        ),
        dict(type="buttons", direction="left", x=0, y=-0.06,
             xanchor="left", yanchor="top",
             
            buttons=[
                dict(label="▶ Play", method="animate", args=[None, {
                        "frame": {"duration": 800, "redraw": True},
                        "fromcurrent": True,
                        "transition": {"duration": 300}
                    }]
                ),
                dict(label="〓 Pause", method="animate", args=[[None], {
                        "mode": "immediate",
                        "frame": {"duration": 0, "redraw": False}
                    }]
                )
            ],
            bgcolor="white",
            bordercolor="#ccc"
        )
    ],
    sliders=[{"active": 0, "y": 0.0, "x": 0.15, "len": 0.85, "currentvalue": {
            "prefix": "Quarter: ",
            "font": {"size": 14, "color": "#0d6efd"}
        },
        "steps": [
            {
                "args": [[q], {
                    "frame": {"duration": 500, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 300}
                }],
                "label": q,
                "method": "animate"
            } for q in quarters
        ]
    }],
    mapbox=dict(
        style="carto-positron",
        zoom=4,
        center=dict(lat=50, lon=10)
    ),
    margin={"r": 0, "t": 100, "l": 0, "b": 0},
    height=750,
    title=custom_legend_title,
)

fig.show()

output_filename = "Protests_Distribution_Map.html"
pio.write_html(fig, file=output_filename, auto_open=True)

In [ ]:
# Plot 2

import plotly.graph_objects as go
import pandas as pd

# ==============================
# 1. Load Data
# ==============================
df_panel = pd.read_csv('Final_Merged_Panel.csv')
df_panel['year_month'] = pd.to_datetime(df_panel['year_month'])

target_countries = ['Germany', 'France', 'United Kingdom', 'Czech Republic', 'Slovakia']
df_panel = df_panel[df_panel['country'].isin(target_countries)].copy()
df_panel = df_panel.sort_values(['country', 'year_month'])

country_colors = {'Germany': '#1f77b4', 'France': '#d62728', 'United Kingdom': '#2ca02c',
                  'Czech Republic': '#9467bd', 'Slovakia': '#ff7f0e'
}

rgba_map = {'Germany': 'rgba(31, 119, 180, 0.18)', 'France': 'rgba(214, 39, 40, 0.18)',
            'United Kingdom': 'rgba(44, 160, 44, 0.18)', 'Czech Republic': 'rgba(148, 103, 189, 0.18)',
            'Slovakia': 'rgba(255, 127, 14, 0.18)'
}

# ==============================
# 2. Create Figure
# ==============================
fig_r1 = go.Figure()

# A. Five Country Overview Lines
for country in target_countries:
    df_c = df_panel[df_panel['country'] == country]
    fig_r1.add_trace(go.Scatter(
        x=df_c['year_month'],
        y=df_c['crackdown_rate'],
        mode='lines',
        name=country,
        line=dict(width=2.5, color=country_colors[country]),
        hovertemplate=(
            "<b>%{text}</b><br>" +
            "Date: %{x|%Y-%m}<br>" +
            "Repression Rate: %{y:.3f}<extra></extra>"
        ),
        text=[country] * len(df_c),
        visible=True
    ))

# B. Average Line
df_avg = df_panel.groupby('year_month', as_index=False)['crackdown_rate'].mean()
fig_r1.add_trace(go.Scatter(
    x=df_avg['year_month'],
    y=df_avg['crackdown_rate'],
    mode='lines',
    name='Average',
    line=dict(width=3, color='black', dash='dash'),
    hovertemplate=(
        "<b>Overall Average</b><br>" +
        "Date: %{x|%Y-%m}<br>" +
        "Average Rate: %{y:.3f}<extra></extra>"
    ),
    visible=True
))

# C. Single Country Filled Area
for country in target_countries:
    df_c = df_panel[df_panel['country'] == country]
    fig_r1.add_trace(go.Scatter(
        x=df_c['year_month'],
        y=df_c['crackdown_rate'],
        mode='lines',
        name=f"{country} filled",
        line=dict(width=3, color=country_colors[country]),
        fill='tozeroy',
        fillcolor=rgba_map[country],
        hovertemplate=(
            "<b>%{text}</b><br>" +
            "Date: %{x|%Y-%m}<br>" +
            "Repression Rate: %{y:.3f}<extra></extra>"
        ),
        text=[country] * len(df_c),
        visible=False
    ))

# D. Single Country Peak Markers
for country in target_countries:
    df_c = df_panel[df_panel['country'] == country].copy()
    peak_row = df_c.loc[df_c['crackdown_rate'].idxmax()]

    peak_label = f"{peak_row['year_month'].strftime('%Y-%m')}<br>{peak_row['crackdown_rate']:.2f}"

    fig_r1.add_trace(go.Scatter(
        x=[peak_row['year_month']],
        y=[peak_row['crackdown_rate']],
        mode='markers+text',
        name=f"{country} peak",
        marker=dict(
            size=10,
            color=country_colors[country],
            line=dict(color='white', width=1.5)
        ),
        text=[peak_label],
        textposition='top center',
        hovertemplate=(
            f"<b>{country} Peak</b><br>" +
            "Date: %{x|%Y-%m}<br>" +
            "Peak Rate: %{y:.3f}<extra></extra>"
        ),
        visible=False
    ))

# ==============================
# 3. Build Dropdown Menu
# ==============================
buttons = []

# 3.1 All Countries: Five lines + average
visible_all = [True, True, True, True, True, True] + [False] * 10
buttons.append(
    dict(
        label="All Countries",
        method="update",
        args=[
            {"visible": visible_all},
            {
                "title": "<b>Cross-National Repression Rate Trajectories (2020-2025)</b><br><sup>Reveals variation in democratic resilience to dissent</sup>"
            }
        ]
    )
)

# 3.2 Average Only
visible_avg_only = [False, False, False, False, False, True] + [False] * 10
buttons.append(
    dict(
        label="Average Only",
        method="update",
        args=[
            {"visible": visible_avg_only},
            {
                "title": "<b>Five-Country Average Repression Rate Trajectory (2020-2025)</b><br><sup>Benchmark for individual country deviations</sup>"
            }
        ]
    )
)

# 3.3 Single Country: Area + Average + Peak
for i, country in enumerate(target_countries):
    visible_list = [False] * 16
    visible_list[5] = True    
    visible_list[6 + i] = True   
    visible_list[11 + i] = True   

    buttons.append(
        dict(
            label=country,
            method="update",
            args=[
                {"visible": visible_list},
                {
                    "title": f"<b>Figure 2: {country} Repression Rate Trajectory (2020-2025)</b><br><sup>Individual country dynamics vs. five-country average</sup>"
                }
            ]
        )
    )

# ==============================
# 4. Layout
# ==============================
fig_r1.update_layout(
    title=(
    "<span style='font-size:22px;'><b>Cross-National Repression Rate Trajectories (2020-2025)</b></span>"
    "<br><br>"
    "<span style='font-size:22px; color:#666;'><sup>Reveals variation in democratic resilience to dissent</sup></span>"
),
    height=580,
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend_title="Country / Reference",
    margin=dict(l=70, r=40, t=120, b=70),

    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1.02,
            xanchor="left",
            y=1.1,
            yanchor="top",
            bgcolor="white",
            bordercolor="lightgray",
            borderwidth=1,
            font=dict(size=12)
        )
    ]
)

# ==============================
# 5. Axes
# ==============================
fig_r1.update_xaxes(
    title_text="Date (Year-Month)",
    showline=True,
    linecolor='black',
    linewidth=1.2,
    showgrid=True,
    gridcolor='rgba(0,0,0,0.08)',
    ticks='outside',
    tickangle=45,
    tickformat="%Y-%m",
    dtick="M3",
    tickfont=dict(size=11),
    zeroline=False
)

fig_r1.update_yaxes(
    title_text="Repression Rate",
    showline=True,
    linecolor='black',
    linewidth=1.2,
    showgrid=True,
    gridcolor='rgba(0,0,0,0.08)',
    ticks='outside',
    tickfont=dict(size=11),
    rangemode='tozero',
    zeroline=True,
    zerolinecolor='rgba(0,0,0,0.25)',
    zerolinewidth=1
)

fig_r1.show()

output_filename = "Repression_Line_Chat.html"
pio.write_html(fig_r1, file=output_filename, auto_open=True)

In [ ]:
## Plot 3

import pandas as pd
import plotly.graph_objects as go

# ==========================================
# 1. Data preprocessing and annual aggregation
# ==========================================
df_micro = pd.read_csv('Micro_With_VDem.csv')
target_countries = ['Germany', 'France', 'United Kingdom', 'Czech Republic', 'Slovakia']
df_micro = df_micro[df_micro['country'].isin(target_countries)].copy()

# Extract year
df_micro['year'] = pd.to_datetime(df_micro['event_date']).dt.year

# Determine whether intervention has been carried out
df_micro['is_intervened'] = df_micro['sub_event_type'].isin(
    ['Protest with intervention', 'Excessive force against protesters']
).astype(int)

country_colors = {
    'Germany': '#1f77b4',         
    'France': '#17becf',         
    'United Kingdom': '#2ca02c',  
    'Czech Republic': '#5C4033', 
    'Slovakia': '#ff7f0e'      
}

# Identify the locations that need to be preserved (>=2 events throughout the entire time period to maintain map stability)
valid_locs = df_micro.groupby(['country', 'location']).size().reset_index(name='total_count')
valid_locs = valid_locs[valid_locs['total_count'] >= 2][['country', 'location']]
df_filtered = df_micro.merge(valid_locs, on=['country', 'location'], how='inner')

# Aggregate by country location year
agg_df = df_filtered.groupby(['country', 'location', 'latitude', 'longitude', 'year']).agg(
    total_events=('event_id_cnty', 'count'),
    intervened_events=('is_intervened', 'sum')
).reset_index()

agg_df['has_intervention'] = (agg_df['intervened_events'] > 0).astype(int)

years = sorted(agg_df['year'].unique())
y0 = years[0]

# ==========================================
# 2. Build 12 underlying trajectory architectures
# ==========================================
fig = go.Figure()

fig.add_trace(go.Scattermapbox(
    lat=[0], lon=[0], mode='markers',
    name='All Protests (Baseline)',
    marker=dict(size=8, color='#3498db'),
    visible=False, showlegend=True, hoverinfo='skip'
))
fig.add_trace(go.Scattermapbox(
    lat=[0], lon=[0], mode='markers',
    name='State Interventions',
    marker=dict(size=8, color='#E74C3C'),
    visible=False, showlegend=True, hoverinfo='skip'
))

for c in target_countries:
    df_y0_c = agg_df[(agg_df['year'] == y0) & (agg_df['country'] == c)]
    df_y0_red = df_y0_c[df_y0_c['has_intervention'] == 1]

    # baseline：Change to each country's own color
    fig.add_trace(go.Scattermapbox(
        lat=df_y0_c['latitude'],
        lon=df_y0_c['longitude'],
        mode='markers',
        marker=dict(
            size=7,
            color=country_colors[c],
            opacity=0.45
        ),
        showlegend=False,
        customdata=df_y0_c[['location', 'country', 'total_events']],
        hovertemplate="<b>%{customdata[0]}</b><br>Country: %{customdata[1]}<br>Total Events: %{customdata[2]}<extra></extra>",
        visible=False
    ))

    # intervention：Unified Red
    fig.add_trace(go.Scattermapbox(
        lat=df_y0_red['latitude'],
        lon=df_y0_red['longitude'],
        mode='markers',
        marker=dict(
            size=11,
            color='#E74C3C',
            opacity=0.92
        ),
        showlegend=False,
        customdata=df_y0_red[['location', 'country', 'intervened_events']],
        hovertemplate="<b>%{customdata[0]}</b><br>Country: %{customdata[1]}<br><b>Intervened Events: %{customdata[2]}</b><extra></extra>",
        visible=False
    ))

# ==========================================
# 3. Pre calculated timeline frames
# ==========================================
frames = []

for y in years:
    frame_data = []
    frame_data.append(go.Scattermapbox(lat=[0], lon=[0], visible=False))
    frame_data.append(go.Scattermapbox(lat=[0], lon=[0], visible=False))

    for c in target_countries:
        df_y_c = agg_df[(agg_df['year'] == y) & (agg_df['country'] == c)]
        df_y_red = df_y_c[df_y_c['has_intervention'] == 1]

        frame_data.append(go.Scattermapbox(
            lat=df_y_c['latitude'],
            lon=df_y_c['longitude'],
            mode='markers',
            marker=dict(
                size=7,
                color=country_colors[c],
                opacity=0.45
            ),
            customdata=df_y_c[['location', 'country', 'total_events']],
            hovertemplate="<b>%{customdata[0]}</b><br>Country: %{customdata[1]}<br>Total Events: %{customdata[2]}<extra></extra>"
        ))

        # intervention：Unified Red
        frame_data.append(go.Scattermapbox(
            lat=df_y_red['latitude'],
            lon=df_y_red['longitude'],
            mode='markers',
            marker=dict(
                size=11,
                color='#E74C3C',
                opacity=0.92
            ),
            customdata=df_y_red[['location', 'country', 'intervened_events']],
            hovertemplate="<b>%{customdata[0]}</b><br>Country: %{customdata[1]}<br><b>Intervened Events: %{customdata[2]}</b><extra></extra>"
        ))

    frames.append(go.Frame(data=frame_data, name=str(y)))

fig.frames = frames

# ==========================================
# 4. Multiple choice button
# ==========================================
def get_vis(country_list):
    vis = [False, False]
    for c in target_countries:
        vis.extend([True, True] if c in country_list else [False, False])
    return vis

combo_buttons = [
    dict(label="🌐 All Countries", method="restyle", args=[{"visible": get_vis(target_countries)}]),
    dict(label="⚔️ West (UK, FR, DE)", method="restyle", args=[{"visible": get_vis(['United Kingdom', 'France', 'Germany'])}]),
    dict(label="⚔️ East (CZ, SK)", method="restyle", args=[{"visible": get_vis(['Czech Republic', 'Slovakia'])}]),
]

for c in target_countries:
    combo_buttons.append(
        dict(label=f"🔹 Only {c}", method="restyle", args=[{"visible": get_vis([c])}])
    )

# ==========================================
# 5. layout
# ==========================================
fig.update_layout(
    uirevision='locked',
    legend=dict(
        orientation="h",
        y=0.97,
        x=0.5,
        xanchor="center",
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='lightgray',
        borderwidth=1,
        font=dict(size=12)
    ),

    updatemenus=[
        dict(buttons=combo_buttons, direction="down", x=0.0, y=1, xanchor="left",
             yanchor="top", showactive=True, bgcolor="white", bordercolor="#ccc"
        ),
        
        dict(type="buttons", direction="left", x=0, y=-0.07, xanchor="left",
             yanchor="top",
             
            buttons=[
                dict(label="▶ Play", method="animate", args=[None, {
                        "frame": {"duration": 1500, "redraw": True},
                        "fromcurrent": True,
                        "transition": {"duration": 500}
                    }]
                ),
                
                dict(label="〓 Pause", method="animate", args=[[None], {
                        "mode": "immediate",
                        "frame": {"duration": 0, "redraw": False}
                    }]
                )
            ],
             
            bgcolor="white",
            bordercolor="#ccc"
        )
    ],

    sliders=[{"active": 0, "y": 0.05, "x": 0.13, "len": 0.85, "currentvalue": {"prefix": "Year: ", "font": {"size": 16, "color": "#0d6efd"}},
              
        "steps": [
            {
                "args": [[str(y)], {
                    "frame": {"duration": 800, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 300}
                }],
                
                "label": str(y),
                "method": "animate"
            }
            
            for y in years
        ]
    }],

    mapbox=dict(
        style="carto-positron",
        zoom=4,
        center=dict(lat=50, lon=10)
    ),
    
    margin={"r": 0, "t": 100, "l": 0, "b": 0},
    height=700,
    title=dict(
        text="<b>Comparison of Cross-border Space Deterrence (Based on Annual Aggregation)</b><b><br><sup>Different colors represent different countries, with red dots indicating state intervention events</sup>",
        font=dict(size=20),
        y=0.91, 
        x=0.5, 
        xanchor='center'
    )
)

fig.show()

output_filename = "Deterrence_Comparison_Map.html"
pio.write_html(fig, file=output_filename, auto_open=True)

In [ ]:
## Plot 4 & 5

import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ==========================================
# 1. Data preparation
# ==========================================
df_micro = pd.read_csv('Micro_With_VDem.csv')

target_countries = ['Germany', 'France', 'United Kingdom', 'Czech Republic', 'Slovakia']
df_micro = df_micro[df_micro['country'].isin(target_countries)].copy()

df_micro['year'] = pd.to_datetime(df_micro['event_date']).dt.year

capitals = {
    'Germany': 'Berlin',
    'France': 'Paris',
    'United Kingdom': 'London',
    'Czech Republic': 'Prague',
    'Slovakia': 'Bratislava'
}

df_micro['location'] = df_micro['location'].astype(str)
df_micro['capital_name'] = df_micro['country'].map(capitals)

df_micro['is_capital'] = np.where(
    df_micro.apply(
        lambda row: row['capital_name'].lower() in row['location'].lower()
        if pd.notna(row['capital_name']) and pd.notna(row['location']) else False,
        axis=1
    ),
    1, 0
)

df_micro['is_intervened'] = df_micro['sub_event_type'].isin(
    ['Protest with intervention', 'Excessive force against protesters']
).astype(int)

df_micro['Region'] = df_micro['is_capital'].map({1: 'Capital City', 0: 'Other Regions'})

# ==========================================
# 2. Plot 4: Dumbbell chart (overall average)
# ==========================================
df_spatial = (
    df_micro.groupby(['country', 'Region'])
    .agg(
        total=('event_id_cnty', 'count'),
        intervened=('is_intervened', 'sum')
    )
    .reset_index()
)

df_spatial['rate'] = df_spatial['intervened'] / df_spatial['total']

df_wide = df_spatial.pivot(index='country', columns='Region', values='rate').reset_index()
for col in ['Capital City', 'Other Regions']:
    if col not in df_wide.columns:
        df_wide[col] = 0

df_wide = df_wide.fillna(0)
df_wide['gap'] = df_wide['Capital City'] - df_wide['Other Regions']
df_wide['gap_label'] = df_wide['gap'].map(lambda x: f"{x:+.1%}")
df_wide = df_wide.sort_values('gap', ascending=True).reset_index(drop=True)

fig_dumbbell = go.Figure()

for _, row in df_wide.iterrows():
    fig_dumbbell.add_trace(go.Scatter(
        x=[row['Other Regions'], row['Capital City']],
        y=[row['country'], row['country']],
        mode='lines',
        line=dict(color='rgba(120,120,120,0.55)', width=3),
        hoverinfo='skip',
        showlegend=False
    ))

fig_dumbbell.add_trace(go.Scatter(
    x=df_wide['Other Regions'],
    y=df_wide['country'],
    mode='markers+text',
    name='Other Regions',
    marker=dict(size=14, color='#AED6F1', line=dict(color='white', width=1.5)),
    text=[f"{v:.1%}" for v in df_wide['Other Regions']],
    textposition='middle left',
    customdata=df_wide[['gap_label']],
    hovertemplate="<b>%{y}</b><br>Other Regions: %{x:.2%}<br>Gap: %{customdata[0]}<extra></extra>"
))

fig_dumbbell.add_trace(go.Scatter(
    x=df_wide['Capital City'],
    y=df_wide['country'],
    mode='markers+text',
    name='Capital City',
    marker=dict(size=16, color='#E74C3C', line=dict(color='white', width=1.5)),
    text=[f"{v:.1%}" for v in df_wide['Capital City']],
    textposition='middle right',
    customdata=df_wide[['gap_label']],
    hovertemplate="<b>%{y}</b><br>Capital City: %{x:.2%}<br>Gap: %{customdata[0]}<extra></extra>"
))

fig_dumbbell.add_trace(go.Scatter(
    x=(df_wide['Other Regions'] + df_wide['Capital City']) / 2,
    y=df_wide['country'],
    mode='text',
    text=df_wide['gap_label'],
    textposition='top center',
    textfont=dict(size=11, color='dimgray'),
    showlegend=False,
    hoverinfo='skip'
))

fig_dumbbell.update_layout(
    title="<b>Capital —— Local: The Violence Gap</b><br>"
    "<sup>The longer the line segment, the more concentrated the<br>"
    "state's coercive intervention is at the political center</sup>",
    xaxis_title="Mandatory intervention rate",
    yaxis_title="Countries",
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=560,
    margin=dict(l=120, r=60, t=120, b=70),
    legend=dict(orientation='h', y=1.08, x=0.5, xanchor='center')
)

fig_dumbbell.update_xaxes(
    tickformat=".0%",
    showgrid=True,
    gridcolor='rgba(0,0,0,0.08)',
    zeroline=False
)
fig_dumbbell.update_yaxes(
    showgrid=False,
    categoryorder='array',
    categoryarray=df_wide['country']
)

fig_dumbbell.show()

output_filename = "Violence_Gap_Dumbbell_Chart.html"
pio.write_html(fig_dumbbell, file=output_filename, auto_open=True)

In [ ]:
# ==========================================
# 3. Plot 5 Annual Gap Trend Chart 
# ==========================================
df_year = (
    df_micro.groupby(['country', 'year', 'Region'])
    .agg(
        total=('event_id_cnty', 'count'),
        intervened=('is_intervened', 'sum')
    )
    .reset_index()
)

df_year['rate'] = df_year['intervened'] / df_year['total']

df_year_wide = df_year.pivot(index=['country', 'year'], columns='Region', values='rate').reset_index()
for col in ['Capital City', 'Other Regions']:
    if col not in df_year_wide.columns:
        df_year_wide[col] = 0

df_year_wide = df_year_wide.fillna(0)
df_year_wide['Gap'] = df_year_wide['Capital City'] - df_year_wide['Other Regions']

years = sorted(df_year_wide['year'].unique())

def subset_country(country_name):
    if country_name == 'All Countries':
        temp = (
            df_year_wide.groupby('year')[['Capital City', 'Other Regions', 'Gap']]
            .mean()
            .reset_index()
        )
    else:
        temp = df_year_wide[df_year_wide['country'] == country_name].copy()
    return temp.sort_values('year')

initial_country = 'All Countries'
df_init = subset_country(initial_country)

fig_gap = go.Figure()

# 0: Capital City
fig_gap.add_trace(go.Scatter(
    x=df_init['year'],
    y=df_init['Capital City'],
    mode='lines+markers',
    name='Capital City',
    line=dict(color='#E74C3C', width=3),
    marker=dict(size=8),
    hovertemplate="Year: %{x}<br>Capital City: %{y:.2%}<extra></extra>"
))

# 1: Other Regions
fig_gap.add_trace(go.Scatter(
    x=df_init['year'],
    y=df_init['Other Regions'],
    mode='lines+markers',
    name='Other Regions',
    line=dict(color='#3498DB', width=3),
    marker=dict(size=8),
    hovertemplate="Year: %{x}<br>Other Regions: %{y:.2%}<extra></extra>"
))

# 2: Gap
fig_gap.add_trace(go.Scatter(
    x=df_init['year'],
    y=df_init['Gap'],
    mode='lines+markers',
    name='Gap (Capital - Other)',
    line=dict(color='#2C3E50', width=3, dash='dash'),
    marker=dict(size=7),
    hovertemplate="Year: %{x}<br>Gap: %{y:.2%}<extra></extra>"
))

# dropdown
buttons = []
for c in ['All Countries'] + target_countries:
    df_c = subset_country(c)
    buttons.append(dict(
        label=c,
        method='update',
        args=[
            {
                'x': [df_c['year'], df_c['year'], df_c['year']],
                'y': [df_c['Capital City'], df_c['Other Regions'], df_c['Gap']]
            },
            {
                'title': f"<b>Capital —— Local, Annual changes in intervention gap</b><br><sup>Current view：{c}</sup>"
            }
        ]
    ))

fig_gap.update_layout(
    title="<b>Capital —— Local, Annual changes in intervention gap</b><br><sup>Current view：All Countries</sup>",
    xaxis_title="Year",
    yaxis_title="Intervention rate / Difference",
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=560,
    margin=dict(l=80, r=60, t=110, b=70),
    legend=dict(
        orientation='h',
        y=1.08,
        x=0.5,
        xanchor='center'
    ),
    updatemenus=[
        dict(
            buttons=buttons,
            direction='down',
            x=-0.07,
            y=1.1,
            xanchor='left',
            yanchor='top',
            showactive=True,
            bgcolor='white',
            bordercolor='#ccc'
        )
    ],
    shapes=[
        dict(
            type='line',
            x0=min(years),
            x1=max(years),
            y0=0,
            y1=0,
            line=dict(color='gray', width=1.2, dash='dot')
        )
    ]
)

fig_gap.update_xaxes(
    tickmode='linear',
    dtick=1,
    showgrid=False
)

fig_gap.update_yaxes(
    tickformat=".0%",
    showgrid=True,
    gridcolor='rgba(0,0,0,0.08)',
    zeroline=False
)

fig_gap.show()

output_filename = "Intervention_Gap_Line_Chart.html"
pio.write_html(fig_gap, file=output_filename, auto_open=True)

In [ ]:
## Plot 6

import pandas as pd
import plotly.graph_objects as go

# ==========================================
# 1. Data preprocessing: Retrieve all required V-Dem dimensions during aggregation
# ==========================================
df_panel = pd.read_csv('Final_Merged_Panel.csv')
df_panel['year_month'] = pd.to_datetime(df_panel['year_month'])
df_panel['Year'] = df_panel['year_month'].dt.year
df_panel['Quarter'] = df_panel['year_month'].dt.quarter
df_panel['Time_Quarter'] = df_panel['Year'].astype(str) + '-Q' + df_panel['Quarter'].astype(str)

baseline_years = [2021, 2023]
df_baseline = df_panel[df_panel['Year'].isin(baseline_years)].groupby('country')['crackdown_rate'].mean().reset_index()
df_baseline.columns = ['country', 'baseline_rate']

df_quarterly = df_panel.groupby(['country', 'Time_Quarter']).agg(
    current_rate=('crackdown_rate', 'mean'),
    total_events=('total_protests', 'sum'),
    v2x_libdem=('v2x_libdem', 'mean'),
    v2x_rule=('v2x_rule', 'mean'),
    v2x_polyarchy=('v2x_polyarchy', 'mean'),
    v2x_freexp_altinf=('v2x_freexp_altinf', 'mean')
).reset_index()

df_dynamic = pd.merge(df_quarterly, df_baseline, on='country').dropna()

MIN_SIZE = 20
MAX_SIZE = 75
global_min_events = df_dynamic['total_events'].min()
global_max_events = df_dynamic['total_events'].max()

def calculate_smooth_size(val):
    if global_max_events == global_min_events: return (MIN_SIZE + MAX_SIZE) / 2
    normalized = (val - global_min_events) / (global_max_events - global_min_events)
    return MIN_SIZE + (normalized * (MAX_SIZE - MIN_SIZE))

df_dynamic['smooth_size'] = df_dynamic['total_events'].apply(calculate_smooth_size)

quarters = sorted(df_dynamic['Time_Quarter'].unique())
q0 = quarters[0]

hover_cols = ['baseline_rate', 'current_rate', 'total_events', 'v2x_libdem', 'v2x_rule', 'v2x_polyarchy', 'v2x_freexp_altinf']

# ==========================================
# 2. Visual style
# ==========================================
marker_base = dict(
    line=dict(width=2.5, color='#2C3E50'),
    opacity=0.80
)

# ==========================================
# 3. Build charts
# ==========================================
hover_template = """
<span style="font-size:11px;">--- <b>Crisis Data</b> ---</span><br>
Baseline: %{customdata[0]:.4f} | Current Q: %{customdata[1]:.4f}<br>
Protest Volume: %{customdata[2]:.0f}<br>
<span style="font-size:11px;">--- <b>V-Dem Context</b> ---</span><br>
Liberal Dem: %{customdata[3]:.3f} | Rule of Law: %{customdata[4]:.3f}<br>
Electoral: %{customdata[5]:.3f} | Free Exp: %{customdata[6]:.3f}
<extra></extra>
"""

fig = go.Figure()

for i, c in enumerate(df_dynamic['country'].unique()):
    df_q0_c = df_dynamic[(df_dynamic['Time_Quarter'] == q0) & (df_dynamic['country'] == c)]
    
    m_style = marker_base.copy()
    m_style['size'] = df_q0_c['smooth_size'].values[0]
    
    fig.add_trace(go.Scatter(
        x=[df_q0_c['baseline_rate'].values[0]], 
        y=[df_q0_c['current_rate'].values[0]],
        mode='markers+text', text=c, textposition='top center', textfont=dict(size=12),
        marker=m_style, name=c,
        customdata=df_q0_c[hover_cols],
        hovertemplate=hover_template 
    ))

# ==========================================
# 4. Pre calculate animation frames
# ==========================================
frames = []
for q in quarters:
    frame_data = []
    for c in df_dynamic['country'].unique():
        df_q_c = df_dynamic[(df_dynamic['Time_Quarter'] == q) & (df_dynamic['country'] == c)]
        m_style_frame = marker_base.copy()
        
        if not df_q_c.empty:
            m_style_frame['size'] = df_q_c['smooth_size'].values[0]
            frame_data.append(go.Scatter(
                x=[df_q_c['baseline_rate'].values[0]], 
                y=[df_q_c['current_rate'].values[0]],
                marker=m_style_frame,
                customdata=df_q_c[hover_cols], 
                hovertemplate=hover_template
            ))
        else:
            frame_data.append(go.Scatter(x=[None], y=[None]))
            
    frames.append(go.Frame(data=frame_data, name=q))

fig.frames = frames

# ==========================================
# 5. Draw baseline (manually adjusted to the current result)
# ==========================================
fig.add_shape(type="line", x0=-0.25, y0=-0.3, x1=0.25, y1=0.3, 
              line=dict(color="gray", width=3, dash="dash"))

# ==========================================
# 6. Layout (manually debugged to the current result)
# ==========================================
fig.update_layout(
    uirevision='locked', 
    updatemenus=[
        dict(type="buttons", direction="left", x=-0.025, y=-0.08, xanchor="left", yanchor="top",
             buttons=[
                 dict(label="▶ Play", method="animate", args=[None, {"frame": {"duration": 1200, "redraw": True}, "fromcurrent": True, "transition": {"duration": 400, "easing": "cubic-in-out"}}]),
                 dict(label="〓 Pause", method="animate", args=[[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}}])
             ],
             bgcolor="white", bordercolor="#ccc", font=dict(size=14))
    ],
    sliders=[{
        "active": 0, "y": 0.0, "x": 0.15, "len": 0.85, 
        "currentvalue": {"prefix": "Quarter: ", "font": {"size": 16, "color": "#0d6efd"}},
        "steps": [{"args": [[q], {"frame": {"duration": 600, "redraw": True}, "mode": "immediate", "transition": {"duration": 300}}], "label": q, "method": "animate"} for q in quarters]
    }],
    plot_bgcolor="white",

    xaxis=dict(
        title="<b>Normal Baseline Intervention Rate</b>", 
        linecolor='#AAAAAA', linewidth=1, mirror=True, gridcolor='#E5E7EB', 
        range=[-0.05, 0.25], 
        zeroline=True, zerolinecolor='black', zerolinewidth=2
    ),
    yaxis=dict(
        title="<b>Current Quarter Intervention Rate</b>", 
        linecolor='#AAAAAA', linewidth=1, mirror=True, gridcolor='#E5E7EB', 
        range=[-0.05, 0.3], 
        zeroline=True, zerolinecolor='black', zerolinewidth=2
    ),
    legend=dict(orientation="h", y=1.08, x=0.0, xanchor="left", bgcolor='rgba(255,255,255,0.9)', bordercolor='lightgray', borderwidth=1),

    annotations=[
        dict(x=0.2, y=0.160, 
                text="<b>45° Baseline </b><br>Points on the line = Crisis had no effect<br><span style='color:#c0392b'><b>Points above</b> = Disproportionate Violence Amplification</span>", 
                showarrow=False, 
                font=dict(color="#555555", size=11), 
                align='left'
            )
        ],
    
    title=dict(text="<b>Dynamic Pressure Test with Multi-Dimensional Context</b><br>Hover to inspect V-Dem indices alongside crisis deviations", x=0.5, y=0.98, xanchor='center'),
    height=700, margin=dict(l=70, r=40, t=100, b=60)
)


fig.show()

output_filename = "Dynamic_Pressure_ScatteR_Plot.html"
pio.write_html(fig, file=output_filename, auto_open=True)

In [ ]:
## Plot 7
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ==========================================
# 1. Data Reading and Preprocessing
# ==========================================
df_panel = pd.read_csv('Final_Merged_Panel.csv')
df_panel['year_month'] = pd.to_datetime(df_panel['year_month'])
df_panel['year'] = df_panel['year_month'].dt.year
df_panel['month'] = df_panel['year_month'].dt.month

months_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

countries = sorted(df_panel['country'].unique())
years = sorted(df_panel['year'].unique())
y0 = years[0]

# ==========================================
# 2. Calculate the national normal baseline
# ==========================================
baseline_years = [y for y in years if y not in [2020, 2022]]
df_baseline = (
    df_panel[df_panel['year'].isin(baseline_years)]
    .groupby('country', as_index=False)
    .agg(baseline_rate=('crackdown_rate', 'mean'))
)

df_panel = df_panel.merge(df_baseline, on='country', how='left')
df_panel['deviation_from_baseline'] = df_panel['crackdown_rate'] - df_panel['baseline_rate']

# ==========================================
# 3. Generate annual matrix dictionary
# ==========================================
full_index = pd.MultiIndex.from_product(
    [countries, years, range(1, 13)],
    names=['country', 'year', 'month']
)

df_monthly = (
    df_panel.groupby(['country', 'year', 'month'], as_index=False)
    .agg(
        crackdown_rate=('crackdown_rate', 'mean'),
        baseline_rate=('baseline_rate', 'mean')
    )
)

df_monthly = (
    df_monthly.set_index(['country', 'year', 'month'])
    .reindex(full_index)
    .reset_index()
)

df_monthly['baseline_rate'] = df_monthly.groupby('country')['baseline_rate'].transform(lambda x: x.ffill().bfill())
df_monthly['crackdown_rate'] = df_monthly['crackdown_rate'].fillna(0)
df_monthly['deviation_from_baseline'] = df_monthly['crackdown_rate'] - df_monthly['baseline_rate']

# ==========================================
# 4. Generate annual matrix dictionary
# ==========================================
yearly_matrices = {}
yearly_hover = {}

for y in years:
    df_y = df_monthly[df_monthly['year'] == y]

    matrix = (
        df_y.pivot(index='country', columns='month', values='deviation_from_baseline')
        .reindex(index=countries, columns=range(1, 13))
    )

    raw_matrix = (
        df_y.pivot(index='country', columns='month', values='crackdown_rate')
        .reindex(index=countries, columns=range(1, 13))
    )

    baseline_matrix = (
        df_y.pivot(index='country', columns='month', values='baseline_rate')
        .reindex(index=countries, columns=range(1, 13))
    )

    yearly_matrices[y] = matrix.values

    hover_stack = np.dstack([
        raw_matrix.values,
        baseline_matrix.values,
        matrix.values
    ])
    yearly_hover[y] = hover_stack

all_dev = np.concatenate([yearly_matrices[y].flatten() for y in years])
all_dev = all_dev[~np.isnan(all_dev)]
abs_limit = np.nanpercentile(np.abs(all_dev), 95)
zmin, zmax = -abs_limit, abs_limit

# ==========================================
# 5. Building chart
# ==========================================
fig = go.Figure()

fig.add_trace(go.Heatmap(
    z=yearly_matrices[y0],
    x=months_labels,
    y=countries,
    customdata=yearly_hover[y0],
    colorscale='RdBu_r',   
    zmid=0,
    zmin=zmin,
    zmax=zmax,
    colorbar=dict(
        title="Deviation from Baseline",
        tickformat=".3f"
    ),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Month: %{x}<br>"
        "Current rate: %{customdata[0]:.4f}<br>"
        "Baseline rate: %{customdata[1]:.4f}<br>"
        "<b>Deviation: %{customdata[2]:+.4f}</b><extra></extra>"
    )
))

# ==========================================
# 6. Definition of crisis year annotation
# ==========================================
crisis_notes = {
    2020: "COVID-19 shock year: watch for sudden positive deviations in spring.",
    2022: "Russia-Ukraine War shock year: watch for renewed deviations from normal restraint.",
    2023: "Energy crisis year: Lack of red indicates 'Baseline Reset' (prior shocks normalized high violence)."
}

# ==========================================
# 7. Animation frames
# ==========================================
frames = []
for y in years:
    note_text = crisis_notes.get(y, "Normal observation year: deviations should fluctuate closely around zero.")
    # If it is a crisis year, set the font to a prominent red color; Otherwise, it will be gray
    font_color = '#c0392b' if y in [2020, 2022] else '#7f8c8d'  # 2023 is also set to gray because its focus is on 'not red'
    
    frames.append(go.Frame(
        data=[go.Heatmap(
            z=yearly_matrices[y],
            customdata=yearly_hover[y]
        )],
        
        layout=dict(
            annotations=[
                dict(
                    x=1.0, y=1.14,
                    xref='paper', yref='paper',
                    text=f"<b>Insight ({y}):</b> {note_text}",
                    showarrow=False,
                    xanchor='right',
                    font=dict(size=12, color=font_color)
                )
            ]
        ),
        name=str(y)
    ))
fig.frames = frames

# ==========================================
# 8. Layout and controls
# ==========================================
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            x=-0.11,
            y=-0.1,
            xanchor="left",
            yanchor="top",
            buttons=[
                dict(label="▶ Play Years", method="animate", args=[None, {"frame": {"duration": 1500, "redraw": True}, "fromcurrent": True, "transition": {"duration": 400}}]),
                dict(label="〓 Pause", method="animate", args=[[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}}])
            ],
            bgcolor="white",
            bordercolor="#ccc"
        )
    ],
    sliders=[{
        "active": 0, "y": 0.0, "x": 0.15, "len": 0.85,
        "currentvalue": {"prefix": "Year: ", "font": {"size": 16, "color": "#0d6efd"}},
        "steps": [{"args": [[str(y)], {"frame": {"duration": 800, "redraw": True}, "mode": "immediate"}], "label": str(y), "method": "animate"} for y in years]
    }],
    plot_bgcolor="white",
    xaxis=dict(title="<b>Month</b>", linecolor='#2C3E50', linewidth=1, side='top'),
    yaxis=dict(title="<b>Country</b>", linecolor='#2C3E50', linewidth=1, tickfont=dict(size=13), autorange='reversed'),
    title=dict(
        text="<b>Dynamic Stress-Test Heatmap: Deviations from Normal Patterns</b><br><sup>Red = above baseline; Blue = below baseline. Watch how the top-right insight text changes during crisis years.</sup>",
        x=0.5, y=0.95, xanchor='center'
    ),
 
    height=540,
    margin=dict(l=110, r=40, t=110, b=50)
)

fig.show()

output_filename = "Dynamic_Stress_Heatmap.html"
pio.write_html(fig, file=output_filename, auto_open=True)